In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
from transformers import AutoTokenizer
from datasets import Dataset, load_dataset

from pathlib import Path

In [2]:
data_dir = Path("../data/interim/")
ckpt = "distilbert-base-uncased"

## Tokenize text

In [3]:
tokenizer = AutoTokenizer.from_pretrained(ckpt, use_fast=True)

In [4]:
df = pd.read_csv(data_dir/"wndp-api-ohe.csv")
df.columns = df.columns.str.lower().str.replace(" ", "_")
df.dropna(inplace=True)
df.head()

,patient_id,text,terms,clinically_healthy,dermatologic_disease,gastrointestinal_disease,hematologic_disease,neurologic_disease,nonspecific,nutritional_disease,ocular_disease,physical_injury,respiratory_disease,urogenital_disease
0,2224052,"Orphaned. Body: fleas, flea dirt",['Clinically healthy'],1,0,0,0,0,0,0,0,0,0,0
1,554625,can't fly. expired shortly after intake,['Nonspecific'],0,0,0,0,0,1,0,0,0,0,0
2,2683764,Orphaned. Body: ECTOPALASHES - FLEAS,['Clinically healthy'],1,0,0,0,0,0,0,0,0,0,0
3,979650,Collision - Car. Fx- Humerus (L) ; Nutritional...,"['Neurologic disease', 'Nutritional disease', ...",0,0,0,0,1,0,1,0,1,0,0
4,1942311,"NSF, orphaned. NSF, orphaned",['Clinically healthy'],1,0,0,0,0,0,0,0,0,0,0


In [5]:
df.shape

(15636, 14)

In [6]:
sample = df.text[:5]
sample[3]

'Collision - Car. Fx- Humerus (L) ; Nutritional - Emaciation; Trauma - Head. Neurologic: Thought to be a DOA until touching Feathers / Fur / Skin: Punctured where broken humerus is Wings / Arms: Fx - Humerus (L) Compound'

In [7]:
tokenizer(sample[3])

{'input_ids': [101, 12365, 1011, 2482, 1012, 23292, 1011, 20368, 7946, 1006, 1048, 1007, 1025, 28268, 1011, 7861, 6305, 18963, 1025, 12603, 1011, 2132, 1012, 11265, 10976, 27179, 1024, 2245, 2000, 2022, 1037, 2079, 2050, 2127, 7244, 12261, 1013, 6519, 1013, 3096, 1024, 26136, 14890, 2094, 2073, 3714, 20368, 7946, 2003, 4777, 1013, 2608, 1024, 23292, 1011, 20368, 7946, 1006, 1048, 1007, 7328, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [8]:
tokens = tokenizer.tokenize(sample[3])
tokens

['collision',
 '-',
 'car',
 '.',
 'fx',
 '-',
 'hume',
 '##rus',
 '(',
 'l',
 ')',
 ';',
 'nutritional',
 '-',
 'em',
 '##ac',
 '##iation',
 ';',
 'trauma',
 '-',
 'head',
 '.',
 'ne',
 '##uro',
 '##logic',
 ':',
 'thought',
 'to',
 'be',
 'a',
 'do',
 '##a',
 'until',
 'touching',
 'feathers',
 '/',
 'fur',
 '/',
 'skin',
 ':',
 'pun',
 '##cture',
 '##d',
 'where',
 'broken',
 'hume',
 '##rus',
 'is',
 'wings',
 '/',
 'arms',
 ':',
 'fx',
 '-',
 'hume',
 '##rus',
 '(',
 'l',
 ')',
 'compound']

In [9]:
ids = tokenizer.convert_tokens_to_ids(tokens)
ids

[12365,
 1011,
 2482,
 1012,
 23292,
 1011,
 20368,
 7946,
 1006,
 1048,
 1007,
 1025,
 28268,
 1011,
 7861,
 6305,
 18963,
 1025,
 12603,
 1011,
 2132,
 1012,
 11265,
 10976,
 27179,
 1024,
 2245,
 2000,
 2022,
 1037,
 2079,
 2050,
 2127,
 7244,
 12261,
 1013,
 6519,
 1013,
 3096,
 1024,
 26136,
 14890,
 2094,
 2073,
 3714,
 20368,
 7946,
 2003,
 4777,
 1013,
 2608,
 1024,
 23292,
 1011,
 20368,
 7946,
 1006,
 1048,
 1007,
 7328]

In [10]:
tokenizer.decode(ids)

'collision - car. fx - humerus ( l ) ; nutritional - emaciation ; trauma - head. neurologic : thought to be a doa until touching feathers / fur / skin : punctured where broken humerus is wings / arms : fx - humerus ( l ) compound'

In [11]:
tokenizer.decode(tokenizer(sample[3])["input_ids"])

'[CLS] collision - car. fx - humerus ( l ) ; nutritional - emaciation ; trauma - head. neurologic : thought to be a doa until touching feathers / fur / skin : punctured where broken humerus is wings / arms : fx - humerus ( l ) compound [SEP]'

## Build the dataset

In [12]:
# Pick only the columns we need for the model
df = df[list(set(df.columns).difference({"patient_id", "terms"}))]
df.head()

,clinically_healthy,nutritional_disease,text,neurologic_disease,ocular_disease,physical_injury,hematologic_disease,nonspecific,dermatologic_disease,gastrointestinal_disease,respiratory_disease,urogenital_disease
0,1,0,"Orphaned. Body: fleas, flea dirt",0,0,0,0,0,0,0,0,0
1,0,0,can't fly. expired shortly after intake,0,0,0,0,1,0,0,0,0
2,1,0,Orphaned. Body: ECTOPALASHES - FLEAS,0,0,0,0,0,0,0,0,0
3,0,1,Collision - Car. Fx- Humerus (L) ; Nutritional...,1,0,1,0,0,0,0,0,0
4,1,0,"NSF, orphaned. NSF, orphaned",0,0,0,0,0,0,0,0,0


In [13]:
# Split dataset into train, val & test set
_ds = (Dataset
          .from_pandas(df)
          .train_test_split(test_size=0.15, 
                            shuffle=True)
     )
ds = _ds["train"].train_test_split(test_size=0.2)
ds["val"] = ds.pop("test")
ds["test"] = _ds["test"]

In [14]:
ds = ds.remove_columns(column_names=["__index_level_0__"])

In [15]:
labels = sorted(ds["train"].column_names)
labels.remove("text")
labels

['clinically_healthy',
 'dermatologic_disease',
 'gastrointestinal_disease',
 'hematologic_disease',
 'neurologic_disease',
 'nonspecific',
 'nutritional_disease',
 'ocular_disease',
 'physical_injury',
 'respiratory_disease',
 'urogenital_disease']

In [16]:
# Add labels to the dataset as float (pytorch excepts float tensors)
ds = ds.map(lambda row: {"labels": [float(row[l]) for l in labels]})

Map:   0%|          | 0/10632 [00:00<?, ? examples/s]

Map:   0%|          | 0/2658 [00:00<?, ? examples/s]

Map:   0%|          | 0/2346 [00:00<?, ? examples/s]

In [17]:
ds["train"][0]

{'clinically_healthy': 1,
 'nutritional_disease': 0,
 'text': 'No patient info. Too young',
 'neurologic_disease': 0,
 'ocular_disease': 0,
 'physical_injury': 0,
 'hematologic_disease': 0,
 'nonspecific': 0,
 'dermatologic_disease': 0,
 'gastrointestinal_disease': 0,
 'respiratory_disease': 0,
 'urogenital_disease': 0,
 'labels': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]}

Let us look at how long is each of the text description

In [18]:
text_stats = np.array([len(row.split()) if row else 0 for row in ds["train"]["text"]])

In [19]:
#sns.histplot(text_stats, bins=20, kde=True)

In [20]:
#(text_stats < 3).sum()/len(text_stats)

In [21]:
#(text_stats > 128).sum()/len(text_stats)

1% of the data have words more than 128 words

In [22]:
for row in ds["train"]["text"]:
    if len(row.split()) < 3:
        print(row)

Cat contact
Orphan. Healthy
orphaned
sick
Orphaned. Orphaned
Kidnapped. NSF
Orphaned. Healthy
Unknown. NSF
Orphaned. Healthy
NSF
Orphaned. Healthy
Orphaned. IHI
NSF. NSF
Unknown
Onshore. NSF
Orphan
orphaned
Orphaned. Healthy
Not welcome
Orphan. Orphan
Sick/Weak
Undetermined. Undetermined
orphan. orphan
Orphan. Healthy
beached
Juv Displaced
Hatched CRU
Orphan. CH
Orphaned. Non-viable
Beached. Thin
weak
NSF. NSF
Orphaned. Orphan
unknown. NSF
grounded. Salmonellosis
grounded. hypothermia
emaciated. emaciated
Beached
beached
Abandoned. Orphaned
Kidnapped. Healthy
orphan. orphan
Injury. Injury
Displaced. Displaced
sick
Orphaned/Abandoned. Dehydration
ill
nest destroyed
Orphaned. NSF
weak
orphan. orphan
weak
Orphan
downed. Downed
Sick/Weak
wont fly
unknown. orphan
Beached
Orphan
Grounded. Healthy
sick
Orphaned. NSF
Beached. NSF
NSF. NSF
orphaned. orphaned
Trichomonoza. Trichomonosis
Orphaned. Healthy
NSF. NSF
Oiled
Hit window
Orphaned. Healthy
Emaciated. Emaciated
DOA. unknown
Sick. *Poxviru

In [23]:
for row in ds["train"]["text"]:
    if len(row.split()) > 200:
        print(len(row.split()), row)
        print("-" * 20)

236 not moving much, not flighted when approached, appeared injured. B1 - Infectious/systemic disease, B2 - toxicosis. Neurologic: Generalized weakness, ataxia GI / Vent: Melena Feathers / Fur / Skin: Poor feather quality, feather lice Wings / Arms: Small abrasion on left distal phalanges Legs / Feet / Hocks: Small, chronic/necrotic open wounds between webbing of digits 2 and 3 and digits 3 and 4 on both feet. A: Juvenile WOST 24-113 was admitted after being found not moving much and not flying away. PE showed signs of general weakness, ataxia, bad feather quality, feather lice, small abrasion on left distal phalanges, melena, and small chronic, necrotic open wounds on patient's webbing. Suspect systemic disease - infectious vs toxicosis vs other. Guarded prognosis assigned to patient for survival - will depend on patient's response to supportive care.
P: Plan to give patient IVLE 15 ml/kg over one hour followed by Norm-R IVF at 80 ml/kg/d. Will start patient on metronidazole 25 mg/kg 

In [24]:
%%time
def tok_fn(row):
  from transformers import AutoTokenizer
  ckpt = "distilbert-base-uncased"  
  tokenizer = AutoTokenizer.from_pretrained(ckpt)
  return tokenizer(row["text"], 
                  truncation=True, 
                    padding="max_length", 
                    max_length=128)

#def tok_fn(row):
#    return tokenizer(row["text"], 
#                     truncation=True, 
#                     padding="max_length", 
#                     max_length=128)


tok_ds = ds.map(tok_fn, batched=True, num_proc=16, remove_columns=labels + ["text"])

Map (num_proc=16):   0%|          | 0/10632 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/2658 [00:00<?, ? examples/s]

Map (num_proc=16):   0%|          | 0/2346 [00:00<?, ? examples/s]

CPU times: total: 3.02 s
Wall time: 2min 11s


In [25]:
tok_ds

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 10632
    })
    val: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2658
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2346
    })
})

In [26]:
tok_ds["train"][0]

{'labels': [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
 'input_ids': [101,
  2053,
  5776,
  18558,
  1012,
  2205,
  2402,
  102,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0],
 'attention_mask': [1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  

In [27]:
# Save the processed data in a parquet file
for split,split_ds in tok_ds.items():
    split_ds.to_parquet(f"../data/processed/wndp-api-data-{split}.parquet")

Creating parquet from Arrow format:   0%|          | 0/11 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/3 [00:00<?, ?ba/s]

In [28]:
!ls ../data/processed/

'ls' is not recognized as an internal or external command,
operable program or batch file.


In [29]:
%%time
data_files = {
    "train": "../data/processed/wndp-api-data-train.parquet",
    "val": "../data/processed/wndp-api-data-val.parquet",
    "test": "../data/processed/wndp-api-data-test.parquet",
}

ds = load_dataset("parquet", data_files=data_files)

Extracting data files:   0%|          | 0/3 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

CPU times: total: 844 ms
Wall time: 2.61 s
